In [12]:
# Check GPU
!nvidia-smi

Thu Jan 22 01:39:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P0             32W /  250W |    1507MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [13]:
!pip uninstall -y transformers peft accelerate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0


In [14]:
# Install dependencies
!pip install -q transformers peft accelerate evaluate datasets bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [15]:
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_from_disk
import torch
from pathlib import Path
import logging
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from peft import PeftModel
import shutil
import os

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB


In [16]:
# CONFIG 
EPOCHS = 5
LEARNING_RATE = 3e-5
BATCH_SIZE = 8
MAX_LENGTH = 384
DOC_STRIDE = 128

STAGE1_CHECKPOINT = "/kaggle/input/xlm-roberta-stage-1-best/stage1_best"
DATASET_PATH = "/kaggle/input/squad-normalized-for-xlm-roberta/viquad_normalized/viquad_normalized"  
OUTPUT_DIR = "/kaggle/working/stage2_output"
CHECKPOINT_DIR = "/kaggle/working/stage2_checkpoints"
FINAL_MODEL_DIR = "/kaggle/working/stage2_best"

## Load Stage 1 Checkpoint

In [17]:
# Copy sang working directory
TMP_CHECKPOINT = "/kaggle/working/stage1_best"
os.makedirs(TMP_CHECKPOINT, exist_ok=True)

for file in os.listdir(STAGE1_CHECKPOINT):
    src = os.path.join(STAGE1_CHECKPOINT, file)
    dst = os.path.join(TMP_CHECKPOINT, file)
    if os.path.isfile(src):
        shutil.copy(src, dst)

# Load tokenizer từ checkpoint đã copy
tokenizer = AutoTokenizer.from_pretrained(TMP_CHECKPOINT)

# Load base model
base_model = AutoModelForQuestionAnswering.from_pretrained("xlm-roberta-base")

# Load PEFT adapter
model = PeftModel.from_pretrained(base_model, TMP_CHECKPOINT)

# Merge adapter vào model
model = model.merge_and_unload()

for param in model.parameters():
    param.requires_grad = True
print(f"Model parameters: {model.num_parameters():,}")

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model parameters: 277,454,594


## Load ViQuAD Dataset

In [18]:
print(f"Loading ViQuAD from {DATASET_PATH}")
viquad = load_from_disk(DATASET_PATH)

print(f"Train size: {len(viquad['train'])}")
print(f"Validation size: {len(viquad['validation'])}")
print(f"Test size: {len(viquad['test']) if 'test' in viquad else 'N/A'}")

Loading ViQuAD from /kaggle/input/squad-normalized-for-xlm-roberta/viquad_normalized/viquad_normalized
Train size: 28454
Validation size: 3814
Test size: 7301


## Prepare Data

In [19]:
def prepare_train_features(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )
    
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    
    tokenized["start_positions"] = []
    tokenized["end_positions"] = []
    
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        
        # Nếu không có answer, đặt vị trí là CLS token
        if len(answers["answer_start"]) == 0:
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
            continue
        
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        
        # Lấy sequence_ids để xác định context
        sequence_ids = tokenized.sequence_ids(i)
        
        # Tìm vị trí bắt đầu và kết thúc của context
        context_start = sequence_ids.index(1) if 1 in sequence_ids else 0
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1) if 1 in sequence_ids else len(sequence_ids)
        
        # Tìm token start index
        token_start_index = context_start
        while token_start_index <= context_end and offsets[token_start_index][0] <= start_char:
            token_start_index += 1
        token_start_index -= 1
        
        # Tìm token end index
        token_end_index = context_end
        while token_end_index >= context_start and offsets[token_end_index][1] >= end_char:
            token_end_index -= 1
        token_end_index += 1
        
        if (token_start_index < context_start or 
            token_end_index > context_end or
            token_start_index >= len(offsets) or
            token_end_index >= len(offsets) or
            token_start_index < 0 or
            token_end_index < 0):
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        elif not (offsets[token_start_index][0] <= start_char and 
                  offsets[token_end_index][1] >= end_char):
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        else:
            tokenized["start_positions"].append(token_start_index)
            tokenized["end_positions"].append(token_end_index)
    
    return tokenized

print("Tokenizing datasets")
train_dataset = viquad['train'].map(
    prepare_train_features,
    batched=True,
    remove_columns=viquad['train'].column_names,
    desc="Tokenizing train",
    keep_in_memory=True  # tránh lỗi read-only
)

val_dataset = viquad['validation'].map(
    prepare_train_features,
    batched=True,
    remove_columns=viquad['validation'].column_names,
    desc="Tokenizing validation",
    keep_in_memory=True #tránh lỗi read-only
)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

Tokenizing datasets


Tokenizing train:   0%|          | 0/28454 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/3814 [00:00<?, ? examples/s]

Train dataset: 30399 samples
Val dataset: 3937 samples


## Training

In [20]:
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="steps",  
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    gradient_accumulation_steps=4,
    dataloader_num_workers=2,
    report_to="none",
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Starting Stage 2 Training (VI Fine-tune)")

trainer.train()

/tmp/ipykernel_55/2796609781.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting Stage 2 Training (VI Fine-tune)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss,Validation Loss
500,1.822300,1.776547
1000,1.421200,1.603311
1500,1.371200,1.507959
2000,1.060700,1.470507
2500,1.093400,1.508918
3000,0.841500,1.566084
3500,0.901100,1.491798


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

TrainOutput(global_step=3500, training_loss=1.3355552738734655, metrics={'train_runtime': 4619.85, 'train_samples_per_second': 32.9, 'train_steps_per_second': 1.028, 'total_flos': 2.1948339648121344e+16, 'train_loss': 1.3355552738734655, 'epoch': 3.6842105263157894})

## Save Final Model

In [21]:
print(f"Saving best model to {FINAL_MODEL_DIR}")
print("Note: Trainer has already loaded the best checkpoint (lowest eval_loss)")
print(f"Best model will be saved to: {FINAL_MODEL_DIR}")

model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Best model saved to: {FINAL_MODEL_DIR}")
print("Stage 2 completed!")


Saving best model to /kaggle/working/stage2_best
Note: Trainer has already loaded the best checkpoint (lowest eval_loss)
Best model will be saved to: /kaggle/working/stage2_best
Best model saved to: /kaggle/working/stage2_best
Stage 2 completed!


In [22]:
# Check output files
!ls -lh /kaggle/working/stage2_best/

total 1.1G
-rw-r--r-- 1 root root  671 Jan 22 02:57 config.json
-rw-r--r-- 1 root root 1.1G Jan 22 02:57 model.safetensors
-rw-r--r-- 1 root root 4.9M Jan 22 02:57 sentencepiece.bpe.model
-rw-r--r-- 1 root root  964 Jan 22 02:57 special_tokens_map.json
-rw-r--r-- 1 root root 1.4K Jan 22 02:57 tokenizer_config.json
-rw-r--r-- 1 root root  17M Jan 22 02:57 tokenizer.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Quick Evaluation (Optional)

In [24]:
test_examples = viquad['validation'].select(range(10))

for i, example in enumerate(test_examples):
    print(f"\n{'='*60}")
    print(f"Example {i+1}")
    print(f"{'='*60}")
    print(f"Question: {example['question']}")
    print(f"Context: {example['context'][:200]}...")
    
    # Check if answer exists
    if len(example['answers']['text']) > 0:
        print(f"Ground Truth: {example['answers']['text'][0]}")
    else:
        print(f"Ground Truth: [NO ANSWER]")
    
    # Predict
    inputs = tokenizer(
        example['question'],
        example['context'],
        return_tensors="pt",
        truncation="only_second",
        max_length=384,
        padding=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits)
    
    answer_tokens = inputs["input_ids"][0][start_idx:end_idx + 1]
    prediction = tokenizer.decode(answer_tokens, skip_special_tokens=True)
    
    print(f"Prediction: {prediction}")


Example 1
Question: Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra?
Context: Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có. Vào thế kỷ 10, Paris đã là một trong những thành phố chính của Pháp cù...
Ground Truth: trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí
Prediction: thành phố trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí

Example 2
Question: Vị trí địa lý của Pháp có gì đặc biệt?
Context: Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có. Vào thế kỷ 10, Paris đã là một trong những thành phố chính của Pháp cù...
Ground Truth: [NO ANSWER]
Prediction: Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có

Example 3
Question: 